# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}\n")
print(f"Dataset version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All elements are referenced using their `@id` field. mlcroissant gives access to record sets and fields using the metadata object.

In [ ]:
# List all record sets and their contained fields/columns
if not dataset.metadata.record_sets:
    print("No record sets found in metadata! Please check the schema.")
else:
    for rs in dataset.metadata.record_sets:
        print(f"Record Set: {rs.name} (@id={rs.id})")
        print("  Fields and columns:")
        for field in rs.fields:
            print(f"    - {field.name} (@id={field.id}, type={field.data_type})")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id={col.id}, type={col.data_type})")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** All Croissant fields and columns must be referenced by their `@id`.

First, list the record set `@id`s and select one for extraction. Replace with real `@id`s from the schema as discovered above.

In [ ]:
# Get list of all record set ids
record_sets = [rs.id for rs in getattr(dataset.metadata, "record_sets", [])]
print(f"Found record sets (@ids): {record_sets}\n")

# Choose a record set to explore (as an example, the first record set)
if record_sets:
    selected_record_set = record_sets[0]
    print(f"Selected record set: {selected_record_set}")
    # Load all records for this set
    records = list(dataset.records(record_set=selected_record_set))
    df = pd.DataFrame(records)
    print("Columns in this record set:")
    print(df.columns.tolist())
    df.head()
else:
    print("No record sets found to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Find a numeric field by `@id` (from the earlier overview), e.g., a column like "mw" (molecular weight) or "num_peptides".

In [ ]:
# Pick a numeric field in the selected record set
numeric_field_id = None

# Try to find a likely numeric field (such as 'molecular_weight' or similar) from columns
if record_sets:
    # We will just try common candidates (this would be improved by checking the schema fields)
    candidates = [col for col in df.columns if 'weight' in col or 'MW' in col or 'num' in col or 'count' in col or 'abundance' in col or 'Amount' in col]
    if candidates:
        numeric_field_id = candidates[0]  # Take the first matching field
        print(f"Selected numeric field: {numeric_field_id}")
    else:
        # Otherwise, just pick a column of numeric dtype
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        print(f"Autodetected numeric field: {numeric_field_id}")

    # Only proceed if a numeric field is found
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['float64', 'float32', 'int64', 'int32'] else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field ({numeric_field_id}):")
        print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

        # Try grouping by a likely categorical field
        common_groups = [col for col in df.columns if 'group' in col or 'class' in col or 'category' in col or 'type' in col or 'sample' in col]
        group_field = common_groups[0] if common_groups else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} and mean of numeric columns:")
            print(grouped_df.head())
        else:
            print("No categorical group field found for grouping.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if record_sets and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field was found, show boxplot
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10, 4))
        df.boxplot(column=numeric_field_id, by=group_field, grid=False)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Mass Spectrometry EV dataset contains multiple record sets, each with detailed fields referenced by unique `@id`s.
- Example numeric fields were filtered and normalized for downstream analysis.
- The dataset is accessible and ready for data science workflows via the Croissant schema and `mlcroissant` Python library.